In [9]:
import os
from qdrant_client import QdrantClient
from qdrant_client.http.models import Filter, FieldCondition, MatchText
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# === Configuration ===
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "test_collection_oai"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Check that all necessary environment variables are set
if not all([QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY]):
    print("Error: Please set QDRANT_URL, QDRANT_API_KEY, and OPENAI_API_KEY in your .env file.")
    exit()

# === Initialize Clients ===
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

def get_embedding(text: str) -> list[float]:
    """Generates an embedding for a given text using OpenAI's model."""
    try:
        response = openai_client.embeddings.create(
            input=text,
            model="text-embedding-3-small"
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return []

def manual_search(query_text: str):
    """Performs a manual search on the Qdrant collection."""
    print(f"Searching for: '{query_text}'...")

    # Generate the vector for the query
    query_vector = get_embedding(query_text)
    if not query_vector:
        print("Could not generate an embedding for the query. Aborting search.")
        return

    # --- Perform a simple semantic search using the query vector ---
    print("\n--- Performing Semantic Search ---")
    semantic_results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector, # Pass the vector directly
        limit=3
    )

    if not semantic_results:
        print("Semantic search returned no results.")
    else:
        print(f"Found {len(semantic_results)} semantic matches:")
        for i, result in enumerate(semantic_results):
            print(f"  Result {i+1} (Score: {result.score:.2f}):")
            print(f"    Text: {result.payload['text'][:100].replace('\n', ' ')}...")
            print(f"    Source: {result.payload.get('file_name', 'N/A')} (Page {result.payload.get('page_number', 'N/A')})")

    # --- Perform a keyword search with filtering ---
    print("\n--- Performing Keyword Search (MatchText) ---")
    text_results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=Filter(
            must=[
                FieldCondition(
                    key="text",
                    match=MatchText(text=query_text)
                )
            ]
        ),
        limit=3
    )

    if not text_results:
        print("Keyword search returned no results.")
    else:
        print(f"Found {len(text_results)} keyword matches:")
        for i, result in enumerate(text_results):
            print(f"  Result {i+1} (Score: {result.score:.2f}):")
            print(f"    Text: {result.payload['text'][:100].replace('\n', ' ')}...")
            print(f"    Source: {result.payload.get('file_name', 'N/A')} (Page {result.payload.get('page_number', 'N/A')})")


# === Run the searches ===
if __name__ == "__main__":
    query = "Star clusters in the M33 galaxy" 
    manual_search(query)

Searching for: 'Star clusters in the M33 galaxy'...

--- Performing Semantic Search ---


TypeError: object of type 'QueryResponse' has no len()